# 03 — Summarizing the SST sample

**Purpose.** Run map-reduce summarization over all 2,352 hearings in the
space, science and technology sample. One row per hearing, one summary each.

**Sharded, ten ways, without overlap.** `iloc[SHARD::N_SHARDS]` partitions the
index: shard *i* takes every tenth hearing starting at *i*, so the ten shards are
disjoint and together cover the sample exactly. Measured on this sample, that
stride leaves the ten shards within **4 percent** of each other by word count —
9,147k to 9,544k — because 2,352 hearings spread over ten shards is already large
enough for the length distribution to even out. Assigning longest-first to the
least-loaded shard closes that gap to 1 percent and saves six minutes on a
five-hour job, which is not worth the extra machinery.

**Resumable.** A hearing counts as done when its call log contains a `reduce`
line. Each shard reads its own log at startup and skips what is already there, so
a task that hits the wall clock is re-submitted rather than restarted, and
re-running a finished shard costs one directory listing.

**Sizing.** Measured at **4.66 seconds per thousand words** for
`Qwen/Qwen3.5-4B` on this hardware, over five hearings spanning the 10th to 90th
percentile of length. The sample is 93,644k words, so the whole run is about
121 GPU-hours, or **roughly 12 hours per shard** across ten. The wall clock is
set to 20 hours: the estimate plus margin for a slow node and for the model load.

**Model.** `Qwen/Qwen3.5-4B`, bf16, no quantization. Chosen on a five-hearing
comparison against three alternatives: it produced 57 checkable claims — named
people, figures, statutes — against 38, 17 and 11 for the others, with none of
the four asserting anything absent from the transcript. It is not the fastest;
`Llama-3.2-3B` runs at 1.93 s/kword but asserted 11 checkable claims across five
hearings and wrote "the senators expressed concerns" where the others named who
said what. A summary that feeds topic extraction is worth more when it carries
attributable content.

**Inputs.** `data/raw/02_hearings/hearings_sst.parquet` — 2,352 hearings, from `05`.

**Outputs**, all under `interim/summaries/sst_{model}/` on scratch:

| File | Contents |
|---|---|
| `status/{packageId}.jsonl` | every model call: stage, level, input hash, timings, full output |
| `shards/summaries_{shard}.parquet` | one row per hearing this shard finished |

`04` consolidates the shards into the single table under `data/cleaned/`.

## Setup

Every parameter arrives as an environment variable set by the job script, so the
notebook is the same file on all ten tasks and the shard it ran is recorded in
its own output rather than edited into the source.

In [ ]:
import hashlib
import json
import os
import sys
import time
from pathlib import Path

sys.path.insert(0, "../lib")

from paths import HEARINGS_REPO, HF_CACHE, summary_run_dir

HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE)
for line in (Path(".env").read_text().splitlines() if Path(".env").exists() else []):
    if line.startswith("HF_TOKEN="):
        os.environ["HF_TOKEN"] = line.split("=", 1)[1].strip()

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import summarize as S
from chunking import chunk_turns

MODEL = os.environ.get("MODEL", "Qwen/Qwen3.5-4B")
SHARD = int(os.environ.get("SHARD", 0))
N_SHARDS = int(os.environ.get("N_SHARDS", 1))
CHUNK_TOKENS = int(os.environ.get("CHUNK_TOKENS", 6144))
RUN_NAME = os.environ.get("RUN_NAME", f"sst_{MODEL.split('/')[-1]}")

RUN_DIR = summary_run_dir(RUN_NAME)
STATUS_DIR = RUN_DIR / "status"
SHARD_DIR = RUN_DIR / "shards"
SHARD_DIR.mkdir(parents=True, exist_ok=True)

print(f"run     : {RUN_NAME}")
print(f"model   : {MODEL}")
print(f"shard   : {SHARD} of {N_SHARDS}")
print(f"chunk   : {CHUNK_TOKENS:,} tokens")
print(f"gpu     : {torch.cuda.get_device_name(0)}, "
      f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")
print(f"output  : {RUN_DIR}")
print(f"sampling: {S.GENERATION}   prompt {S.PROMPT_VERSION}")

## Step 1 — This shard's hearings

The stride is taken on the file's own order, which `05` fixed by sorting on
`(year, packageId)`. That order is a property of the stored table rather than of
this run, so shard *i* selects the same hearings on every re-submission — which
is what makes resuming safe.

In [ ]:
sst = pd.read_parquet(HEARINGS_REPO / "hearings_sst.parquet")
work = sst.iloc[SHARD::N_SHARDS].reset_index(drop=True)

# A hearing is finished when its log records a reduce call.
done = set()
for path in STATUS_DIR.glob("*.jsonl"):
    for line in path.read_text().splitlines():
        if line.strip() and json.loads(line).get("stage") == "reduce":
            done.add(path.stem)
            break

todo = work.loc[~work["packageId"].isin(done)].reset_index(drop=True)

print(f"sample      {len(sst):>6,} hearings, {sst['n_words'].sum() / 1e6:.1f}M words")
print(f"this shard  {len(work):>6,} hearings, {work['n_words'].sum() / 1e6:.1f}M words")
print(f"  already done {len(work) - len(todo):>5,}")
print(f"  to process   {len(todo):>5,}")
if len(todo):
    print(f"\nprojected: {todo['n_words'].sum() / 1000 * 4.66 / 3600:.1f} h "
          f"at the measured 4.66 s/kword")

## Step 2 — Load the model

In [ ]:
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype="auto", device_map="cuda")
model.eval()
print(f"loaded in {time.time() - t0:.0f}s — "
      f"{sum(p.numel() for p in model.parameters()) / 1e9:.2f}B params, "
      f"{next(model.parameters()).dtype}, "
      f"{torch.cuda.memory_allocated() / 1e9:.1f} GB on device")

## Step 3 — Summarize

The shard parquet is rewritten after every hearing rather than once at the end.
A task killed at the wall clock then leaves a readable file covering everything
it finished, instead of an empty one — and the call logs, which are appended and
flushed per call, are already complete up to the moment it stopped.

Temperature-0.1 sampling is near-greedy but still stochastic, so the generator
is seeded per hearing before its first call. The seed derives from the
`packageId` rather than the loop index: it survives resharding and resumption,
so a hearing re-run after a crash — or in a differently sized array — draws the
same completions.

In [ ]:
SHARD_FILE = SHARD_DIR / f"summaries_{SHARD:02d}.parquet"
rows = []
if SHARD_FILE.exists():
    rows = pd.read_parquet(SHARD_FILE).to_dict("records")

started = time.time()
for n, hearing in enumerate(todo.itertuples(), 1):
    chunks = chunk_turns(hearing.turns, tokenizer, CHUNK_TOKENS, spoken_only=True)
    log = S.CallLog(STATUS_DIR / f"{hearing.packageId}.jsonl",
                    hearing.packageId, MODEL)
    # Deterministic per document, independent of shard layout and resume point.
    torch.manual_seed(int.from_bytes(
        hashlib.sha256(hearing.packageId.encode()).digest()[:4], "big"))
    t0 = time.time()
    try:
        out = S.map_reduce([c["text"] for c in chunks], model, tokenizer,
                           budget_tokens=CHUNK_TOKENS, log=log)
        status, summary = "ok", out["final_summary"]
    except Exception as exc:                                   # noqa: BLE001
        status, summary = f"error: {type(exc).__name__}: {str(exc)[:160]}", ""
        out = {"final_tokens": 0, "n_chunks": len(chunks), "n_calls": len(log.records),
               "n_collapse_rounds": 0}

    rows.append({
        "packageId": hearing.packageId, "model": MODEL,
        "prompt_version": S.PROMPT_VERSION, "shard": SHARD,
        "n_words": int(hearing.n_words), "n_turns": int(hearing.n_turns),
        "n_chunks": out["n_chunks"], "n_calls": out["n_calls"],
        "n_collapse_rounds": out["n_collapse_rounds"],
        "summary_tokens": out["final_tokens"],
        "seconds": round(time.time() - t0, 1),
        "status": status, "summary": summary,
    })
    pd.DataFrame(rows).to_parquet(SHARD_FILE, compression="zstd", index=False)

    if n % 10 == 0 or n == len(todo):
        rate = sum(r["n_words"] for r in rows[-n:]) / 1000 / (time.time() - started)
        left = (todo["n_words"].iloc[n:].sum() / 1000 / rate / 3600) if rate else 0
        print(f"  {n:>4}/{len(todo)}  {hearing.packageId:<20} "
              f"{out['n_chunks']:>3}ch {out['n_calls']:>3}calls  "
              f"{rate:.2f} kword/s  eta {left:.1f} h")

print(f"\nshard {SHARD}: {len(rows):,} hearings in {(time.time() - started) / 3600:.2f} h")
print(pd.DataFrame(rows)["status"].str.split(":").str[0].value_counts().to_string())

## Verification

Run against the shard's own file, re-read from disk. Nothing here inspects the
other nine shards: each task verifies what it produced, and `04` checks the
sample is complete once all ten have finished.

In [ ]:
back = pd.read_parquet(SHARD_FILE)
expected = set(work["packageId"])

assert set(back["packageId"]) <= expected, "this shard wrote a hearing outside its slice"
print(f"1) every row belongs to shard {SHARD} ({len(back):,} of {len(work):,})")

assert not back["packageId"].duplicated().any(), "a hearing was summarized twice"
print("2) no hearing summarized twice")

failed = back.loc[back["status"] != "ok"]
print(f"3) failures: {len(failed)}")
for r in failed.head(5).itertuples():
    print(f"     {r.packageId}: {r.status}")

good = back.loc[back["status"] == "ok"]
over = good.loc[good["summary_tokens"] > S.REDUCE_MAX_TOKENS]
assert len(over) == 0, f"{len(over)} summaries exceed the {S.REDUCE_MAX_TOKENS}-token cap"
print(f"4) every summary within {S.REDUCE_MAX_TOKENS} tokens "
      f"(median {good['summary_tokens'].median():.0f}, max {good['summary_tokens'].max()})")

empty = good.loc[good["summary"].str.strip().eq("")]
assert len(empty) == 0, f"{len(empty)} rows marked ok carry no summary"
print("5) no row marked ok is empty")

logged = sum(1 for p in STATUS_DIR.glob("*.jsonl")
             if p.stem in set(back["packageId"]) for _ in [0])
print(f"6) call logs on disk for {logged:,} of this shard's {len(back):,} hearings")